# Task 4: Conditional GAN Optuna Search

Hyperparameter optimization for the cGAN Generator & Discriminator (Plan 6 §4.6 & Plan 7).
Searches G learning rate, D learning rate, batch size, embedding dimension, and $\lambda_{L1}$.

In [ ]:
# 1. Config
N_TRIALS = 12
OPTUNA_EPOCHS = 15
STUDY_NAME = "task4_conditional_gan"
STORAGE_DB = "sqlite:///task4_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import FS2KDataset
from src.models_gan import UNetGenerator, PatchGANDiscriminator
from src.train_loop import train_one_epoch_gan, validate_gan, set_seed

set_seed(SEED)
device = get_device()

In [ ]:
# 3. Objective Function
train_dataset = FS2KDataset(mode="train", seed=SEED)
val_dataset = FS2KDataset(mode="val", seed=SEED)

def objective(trial: optuna.Trial) -> float:
    g_lr = trial.suggest_float("g_lr", 1e-5, 1e-3, log=True)
    d_lr = trial.suggest_float("d_lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    base_channels = trial.suggest_categorical("base_channels", [32, 64])
    embed_dim = trial.suggest_categorical("embed_dim", [4, 8, 16])
    lambda_l1 = trial.suggest_float("lambda_l1", 10.0, 200.0)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    G = UNetGenerator(base_channels=base_channels, embed_dim=embed_dim).to(device)
    D = PatchGANDiscriminator(base_channels=base_channels).to(device)

    optG = torch.optim.Adam(G.parameters(), lr=g_lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=d_lr, betas=(0.5, 0.999))

    best_val_l1 = float("inf")

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_gan(G, D, train_loader, optG, optD, device, lambda_l1=lambda_l1)
        vl = validate_gan(G, val_loader, device)
        if vl["val_l1"] < best_val_l1:
            best_val_l1 = vl["val_l1"]
        trial.report(vl["val_l1"], step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_l1

In [ ]:
# 4. Run Study & Export Config
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

cfg_path = CONFIGS_ROOT / "task4_best_config.yaml"
with open(cfg_path, "w") as f:
    yaml.dump({
        "task": "task4_conditional_gan",
        "best_trial_number": int(study.best_trial.number),
        "best_value": float(study.best_value),
        "params": study.best_params,
    }, f, indent=2)

print(f"Exported to {cfg_path}")